In [ ]:
import sys
import subprocess

def ensure_colab_and_mount(mount_point='/content/drive'):
    try:
        from google.colab import drive
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'google-colab'])
        from google.colab import drive

    drive.mount(mount_point)
    print(f'Google Drive montado em: {mount_point}')

ensure_colab_and_mount()


In [ ]:
from pathlib import Path
import json
import random
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

import tensorflow as tf
from tensorflow.keras import Model
from tensorflow.keras.layers import Conv2D, Conv2DTranspose, Dropout, Input, MaxPooling2D, concatenate, GroupNormalization, Activation, Add
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import binary_crossentropy

WORKSPACE_ROOT = Path('/content/drive/MyDrive/ResUnet')
DRIVE_DATA_ROOT = WORKSPACE_ROOT / 'dataset'
DATA_ROOT = DRIVE_DATA_ROOT if DRIVE_DATA_ROOT.exists() else Path('dataset')
IMG_DIR = DATA_ROOT / 'images'
MSK_DIR = DATA_ROOT / 'masks'
PATCH_SIZE = 256
N_TRAIN = 600
N_VAL = 100
SEED = 23
ARTIFACT_DIR = (WORKSPACE_ROOT / 'artifacts/river256_subset') if WORKSPACE_ROOT.exists() else Path('artifacts/river256_subset')

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

ORIG_COUNTS = {
    'images_npy': len(list(IMG_DIR.glob('*.npy'))),
    'masks_npy': len(list(MSK_DIR.glob('*.npy'))),
}
print(ORIG_COUNTS)


In [ ]:
def extract_patch_id(stem: str):
    try:
        return int(stem.split('_')[-1])
    except ValueError:
        return stem

def list_pairs():
    x_ids = {p.stem: p for p in IMG_DIR.glob('*.npy')}
    y_ids = {p.stem: p for p in MSK_DIR.glob('*.npy')}
    common_ids = sorted(set(x_ids) & set(y_ids), key=extract_patch_id)
    pairs = [(cid, x_ids[cid], y_ids[cid]) for cid in common_ids]
    return pairs

all_pairs = list_pairs()

assert len(all_pairs) == ORIG_COUNTS['images_npy'], f'Pares validos: {len(all_pairs)}'
assert ORIG_COUNTS['images_npy'] == ORIG_COUNTS['masks_npy']

print(all_pairs[:5])

im = np.load(all_pairs[0][1])
mk = np.load(all_pairs[0][2])
assert im.shape[:2] == (PATCH_SIZE, PATCH_SIZE)
assert mk.shape == (PATCH_SIZE, PATCH_SIZE)

print('OK: pares detectados corretamente e dimensoes 256x256 confirmadas.')


In [ ]:
def split_pairs(pairs, n_train, n_val, seed):
    assert n_train + n_val <= len(pairs), 'N_TRAIN + N_VAL maior que total de pares'
    rng = np.random.default_rng(seed)
    idx = rng.permutation(len(pairs))

    train_idx = sorted(idx[:n_train].tolist())
    val_idx = sorted(idx[n_train:n_train + n_val].tolist())

    print(train_idx)
    print(val_idx)

    train_sel = [pairs[i] for i in train_idx]
    val_sel = [pairs[i] for i in val_idx]
    return train_sel, val_sel

train_sel, val_sel = split_pairs(all_pairs, N_TRAIN, N_VAL, SEED + 11)

assert len(train_sel) == N_TRAIN
assert len(val_sel) == N_VAL
assert set(p[0] for p in train_sel).isdisjoint(set(p[0] for p in val_sel))
print('Subset IDs fixos gerados:', len(train_sel), 'train |', len(val_sel), 'val')


In [ ]:
def build_subset(selected_pairs, split_name):
    X, Y, rows = [], [], []

    for cid, x_path, y_path in selected_pairs:
        img = np.load(x_path).astype(np.float32)
        msk = np.load(y_path).astype(np.float32)

        assert img.ndim == 3, f'Imagem invalida para {cid}: {img.shape}'
        assert msk.ndim == 2, f'Mascara invalida para {cid}: {msk.shape}'
        assert img.shape[:2] == (PATCH_SIZE, PATCH_SIZE), f'Imagem fora do tamanho {PATCH_SIZE}: {img.shape}'
        assert msk.shape == (PATCH_SIZE, PATCH_SIZE), f'Mascara fora do tamanho {PATCH_SIZE}: {msk.shape}'

        if img.max() > 1.0:
            img = img / 255.0

        msk = (msk > 0).astype(np.uint8)

        X.append(img.astype(np.float32))
        Y.append(msk[..., None].astype(np.float32))

        rows.append(
            {
                'split': split_name,
                'id': cid,
                'x_path': str(x_path),
                'y_path': str(y_path),
                'x0': 0,
                'y0': 0,
                'patch_size': PATCH_SIZE,
                'mask_ratio': float(msk.mean()),
            }
        )

    X = np.stack(X, axis=0)
    Y = np.stack(Y, axis=0)
    manifest = pd.DataFrame(rows)
    return X, Y, manifest


In [ ]:
x_train, y_train, mf_train = build_subset(train_sel, 'train')
x_val, y_val, mf_val = build_subset(val_sel, 'val')
INPUT_CHANNELS = x_train.shape[-1]

assert x_train.shape == (N_TRAIN, PATCH_SIZE, PATCH_SIZE, INPUT_CHANNELS)
assert y_train.shape == (N_TRAIN, PATCH_SIZE, PATCH_SIZE, 1)
assert x_val.shape == (N_VAL, PATCH_SIZE, PATCH_SIZE, INPUT_CHANNELS)
assert y_val.shape == (N_VAL, PATCH_SIZE, PATCH_SIZE, 1)
assert set(np.unique(y_train)).issubset({0.0, 1.0})
assert set(np.unique(y_val)).issubset({0.0, 1.0})

(ARTIFACT_DIR / 'manifests').mkdir(parents=True, exist_ok=True)
mf_train.to_csv(ARTIFACT_DIR / 'manifests/train_manifest.csv', index=False)
mf_val.to_csv(ARTIFACT_DIR / 'manifests/val_manifest.csv', index=False)

print('x_train:', x_train.shape, '| y_train:', y_train.shape)
print('x_val  :', x_val.shape, '| y_val  :', y_val.shape)
print('mask ratio train (mean):', float(mf_train['mask_ratio'].mean()))
print('mask ratio val   (mean):', float(mf_val['mask_ratio'].mean()))


In [ ]:
BATCH_SIZE = 32

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_ds = val_ds.batch(BATCH_SIZE).cache().prefetch(tf.data.AUTOTUNE)

# steps_per_epoch = int(np.ceil(len(x_train) / BATCH_SIZE))
steps_per_epoch = 0 # Not used for inference
validation_steps = int(np.ceil(len(x_val) / BATCH_SIZE))

print('steps_per_epoch:', steps_per_epoch)
print('validation_steps:', validation_steps)

In [ ]:
def augment_sync(x, y):
    seed_lr = tf.random.uniform([2], maxval=2**31 - 1, dtype=tf.int32)
    seed_ud = tf.random.uniform([2], maxval=2**31 - 1, dtype = tf.int32)

    x = tf.image.stateless_random_flip_left_right(x, seed=seed_lr)
    y = tf.image.stateless_random_flip_left_right(y, seed=seed_lr)

    x = tf.image.stateless_random_flip_up_down(x, seed=seed_ud)
    y = tf.image.stateless_random_flip_up_down(y, seed=seed_ud)

    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    x = tf.image.rot90(x,k)
    y = tf.image.rot90(y,k)

    y = tf.cast(y >= 0.5, tf.float32)
    return x, y

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(
  buffer_size = N_TRAIN,
  seed = SEED,
  reshuffle_each_iteration=True
)
train_ds = train_ds.map(augment_sync, num_parallel_calls=tf.data.AUTOTUNE)
train_ds = train_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
sample_x, sample_y = next(iter(train_ds.take(1)))
print('train sample shapes:', sample_x.shape, sample_y.shape)

assert sample_x.shape[-1] == INPUT_CHANNELS, f'Entrada invalida: esperado {INPUT_CHANNELS} canais, veio {sample_x.shape[-1]}'
assert sample_y.shape[-1] == 1, f'Mascara invalida: esperado 1 canal, veio {sample_y.shape[-1]}'

In [ ]:
def jaccard_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    y_pred_f = tf.clip_by_value(y_pred_f, 1e-7, 1.0 - 1e-7)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)


def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    y_pred_f = tf.clip_by_value(y_pred_f, 1e-7, 1.0 - 1e-7)
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + smooth) / (
        tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth
    )


def dice_coef_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def jaccard_coef_loss(y_true, y_pred):
    return 1.0 - jaccard_coef(y_true, y_pred)

POS_WEIGHT = 2.0

def weighted_bce(y_true, y_pred, pos_weight=POS_WEIGHT):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.clip_by_value(tf.cast(y_pred, tf.float32), 1e-7, 1.0 - 1e-7)
    loss = -(pos_weight * y_true * tf.math.log(y_pred) + (1.0 - y_true) * tf.math.log(1.0 - y_pred))
    return tf.reduce_mean(loss)

def bce_dice_loss(y_true, y_pred):
    bce = weighted_bce(y_true, y_pred)

    dice = dice_coef_loss(y_true, y_pred)
    return bce + dice


def residual_block(x, filters, dropout_rate=0.2):
    shortcut = x

    x = Conv2D(filters, 3, kernel_initializer='he_normal', padding='same', use_bias=False)(x)
    x = GroupNormalization(groups=8)(x)
    x = Activation('elu')(x)

    if dropout_rate > 0:
        x = Dropout(dropout_rate)(x)

    x = Conv2D(filters, 3, kernel_initializer='he_normal', padding='same', use_bias=False)(x)
    x = GroupNormalization(groups=8)(x)

    if shortcut.shape[-1] != filters:
        shortcut = Conv2D(filters, 1, kernel_initializer='he_normal', padding='same', use_bias=False)(shortcut)
        shortcut = GroupNormalization(groups=8)(shortcut)

    x = Add()([x, shortcut])
    x = Activation('elu')(x)
    return x


def build_resunet(input_shape=(128, 128, 3), base_filters=24):
    inputs = Input(shape=input_shape)

    c1 = residual_block(inputs, base_filters)
    p1 = MaxPooling2D((2, 2))(c1)

    c2 = residual_block(p1, base_filters * 2)
    p2 = MaxPooling2D((2, 2))(c2)

    c3 = residual_block(p2, base_filters * 4)
    p3 = MaxPooling2D((2, 2))(c3)

    c4 = residual_block(p3, base_filters * 8)
    p4 = MaxPooling2D((2, 2))(c4)

    bn = residual_block(p4, base_filters * 16)

    u4 = Conv2DTranspose(base_filters * 8, 2, strides=2, padding='same')(bn)
    u4 = concatenate([u4, c4])
    c5 = residual_block(u4, base_filters * 8)

    u3 = Conv2DTranspose(base_filters * 4, 2, strides=2, padding='same')(c5)
    u3 = concatenate([u3, c3])
    c6 = residual_block(u3, base_filters * 4)

    u2 = Conv2DTranspose(base_filters * 2, 2, strides=2, padding='same')(c6)
    u2 = concatenate([u2, c2])
    c7 = residual_block(u2, base_filters * 2)

    u1 = Conv2DTranspose(base_filters, 2, strides=2, padding='same')(c7)
    u1 = concatenate([u1, c1])
    c8 = residual_block(u1, base_filters)

    outputs = Conv2D(1, 1, activation='sigmoid')(c8)
    return Model(inputs, outputs)


BASE_FILTERS = 32

model = build_resunet(
   input_shape=(PATCH_SIZE, PATCH_SIZE, INPUT_CHANNELS),
   base_filters=BASE_FILTERS
)
model.compile(
  optimizer=Adam(learning_rate=1e-4),
   loss=bce_dice_loss,
   metrics=[
       dice_coef,
      jaccard_coef,
       tf.keras.metrics.BinaryAccuracy(name='binary_accuracy'),
       tf.keras.metrics.Precision(name='precision'),
       tf.keras.metrics.Recall(name='recall')
   ],
)

# Load the pre-trained model
# MODEL_PATH = '/content/drive/MyDrive/unet_model/artifacts/river256_subset/models/resunet_256_subset_best.keras'
# model = tf.keras.models.load_model(MODEL_PATH,
    # custom_objects={'bce_dice_loss': bce_dice_loss,
                    # 'dice_coef': dice_coef,
                    # 'jaccard_coef': jaccard_coef})

assert model.input_shape[1:] == (PATCH_SIZE, PATCH_SIZE, INPUT_CHANNELS)
assert model.output_shape[1:] == (PATCH_SIZE, PATCH_SIZE, 1)
model.summary()

In [ ]:
class ForegroundRateCallback(tf.keras.callbacks.Callback):
    def __init__(self, val_data, threshold=0.5):
        super().__init__()
        self.val_data = val_data
        self.threshold = thresold

    def on_epoch_end(self, epoch, logs = None):
        pred_rates = []
        true_rates = []
        for xb, yb in self.val_data:
            preds = self.model(xb, training = False)
            pred_rates.append(tf.reduce_mean(tf.cast(preds >= self.threshold, tf.float32)))
            true_rates.append(tf.reduce_mean(tf.cast(yb, tf.float32)))

            pred_rate = float(tf.reduce_mean(pred_rates)) if pred_rates else 0.0
            true_rate = float(tf.reduce_mean(true_rates)) if true_rates else 0.0
            print(f'- val_fg_rate_pred: {pred_rate:.4f} | val_fg_rate_true: {true_rate:.4f}')

In [ ]:

(ARTIFACT_DIR / 'models').mkdir(parents=True, exist_ok=True)
(ARTIFACT_DIR / 'history').mkdir(parents=True, exist_ok=True)

callbacks = [
    tf.keras.callbacks.TerminateOnNaN(),
    tf.keras.callbacks.EarlyStopping(
        monitor = 'val_dice_coef',
        mode='max',
        patience=12,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_dice_coef',
        mode='max',
        factor=0.5,
        patience=5,
        min_lr=1e-6,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(ARTIFACT_DIR / 'models/resunet_256_subset_best.keras'),
        monitor='val_dice_coef',
        mode = 'max',
        save_best_only=True,
        verbose=1,
    ),
    tf.keras.callbacks.CSVLogger(str(ARTIFACT_DIR / 'history/history.csv')),
]


In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=100,
    callbacks=callbacks,
    verbose=1,
)

In [ ]:
pred_val = model.predict(x_val, batch_size=BATCH_SIZE)

def compute_iou_np(y_true, y_pred, eps=1e-6):
    y_true_b = y_true.astype(bool)
    y_pred_b = y_pred.astype(bool)
    inter = np.logical_and(y_true_b, y_pred_b).sum()
    union = np.logical_or(y_true_b, y_pred_b).sum()
    return (inter + eps) / (union + eps)


thresholds = np.arange(0.30, 0.81, 0.05)
best_t, best_iou = 0.5, -1.0
for t in thresholds:
    yb = (pred_val >= t).astype(np.uint8)
    iou = compute_iou_np(y_val.astype(np.uint8), yb)
    if iou > best_iou:
        best_iou, best_t = float(iou), float(t)

print('Best threshold:', best_t, '| IoU:', best_iou)

(ARTIFACT_DIR / 'metrics').mkdir(parents=True, exist_ok=True)
metrics_payload = {
    'best_threshold': best_t,
    'best_iou': best_iou,
    'train_samples': int(len(x_train)),
    'val_samples': int(len(x_val)),
    'patch_size': PATCH_SIZE,
    'base_filters': BASE_FILTERS,
    'batch_size': BATCH_SIZE,
}
with open(ARTIFACT_DIR / 'metrics/val_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2)


In [ ]:
(ARTIFACT_DIR / 'debug').mkdir(parents=True, exist_ok=True)
pred_bin = (pred_val >= best_t).astype(np.uint8)

n_show = min(6, len(x_val))
fig, axes = plt.subplots(3, 3, figsize=(10, 8))
if n_show == 1:
    axes = np.expand_dims(axes, axis=0)

def get_rgb_image(img_tensor):
    rgb = img_tensor[:, :, [2, 1, 0]]
    rgb_clara = np.clip(rgb, 0, 0.3)
    return rgb_clara / 0.3

for i in range(3):
    random_index = random.randint(0, len(x_val) - 1)

    rgb_image = get_rgb_image(x_val[random_index])

    axes[i, 0].imshow(rgb_image)
    axes[i, 0].set_title('RGB')
    axes[i, 0].axis('off')

    axes[i, 1].imshow(y_val[random_index], cmap='gray', vmin=0, vmax=1)
    axes[i, 1].set_title('GT mask')
    axes[i, 1].axis('off')

    axes[i, 2].imshow(pred_bin[random_index], cmap='gray', vmin=0, vmax=1)
    axes[i, 2].set_title(f'Pred mask')
    axes[i, 2].axis('off')

fig.tight_layout()
sample_plot_path = ARTIFACT_DIR / 'debug/sample_patches_coloridos_2.png'
fig.savefig(sample_plot_path, dpi=150, bbox_inches='tight')
plt.show()

print(f'Debug figure salva em: {sample_plot_path}')

In [ ]:
assert x_train.shape == (N_TRAIN, PATCH_SIZE, PATCH_SIZE, INPUT_CHANNELS)
assert y_train.shape == (N_TRAIN, PATCH_SIZE, PATCH_SIZE, 1)
assert x_val.shape == (N_VAL, PATCH_SIZE, PATCH_SIZE, INPUT_CHANNELS)
assert y_val.shape == (N_VAL, PATCH_SIZE, PATCH_SIZE, 1)

assert set(np.unique(y_train.astype(np.uint8))).issubset({0, 1})
assert set(np.unique(y_val.astype(np.uint8))).issubset({0, 1})

current_counts = {
    'images_npy': len(list(IMG_DIR.glob('*.npy'))),
    'masks_npy': len(list(MSK_DIR.glob('*.npy'))),
}
assert current_counts == ORIG_COUNTS

required_artifacts = [
    ARTIFACT_DIR / 'manifests/train_manifest.csv',
    ARTIFACT_DIR / 'manifests/val_manifest.csv',
    ARTIFACT_DIR / 'models/resunet_256_subset_best.keras',
    ARTIFACT_DIR / 'history/history.csv',
    ARTIFACT_DIR / 'metrics/val_metrics.json',
    ARTIFACT_DIR / 'debug/sample_patches.png',
]
for artifact_path in required_artifacts:
    assert artifact_path.exists(), f'Artifacto faltando: {artifact_path}'

print('Checklist final OK.')


In [ ]:
# Caminho para o arquivo de histórico
history_path = ARTIFACT_DIR / 'history/history.csv'

if history_path.exists():
    history_df = pd.read_csv(history_path)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Gráfico do Coeficiente Dice
    ax1.plot(history_df['epoch'], history_df['dice_coef'], label='Train Dice')
    ax1.plot(history_df['epoch'], history_df['val_dice_coef'], label='Val Dice')
    ax1.set_title('Dice Coefficient over Epochs')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Dice')
    ax1.legend()
    ax1.grid(True)

    # Gráfico do Coeficiente Jaccard (IoU)
    ax2.plot(history_df['epoch'], history_df['jaccard_coef'], label='Train Jaccard')
    ax2.plot(history_df['epoch'], history_df['val_jaccard_coef'], label='Val Jaccard')
    ax2.set_title('Jaccard Coefficient over Epochs')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Jaccard')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()
else:
    print(f'Arquivo de histórico não encontrado em: {history_path}. Certifique-se de que o treinamento foi executado.')